# GT Keyword Extraction and Processing Workflow

In [15]:
# Configuration for German website
dataset_name = "german"
dataset_language = "german"
base_url = f"https://cs.uef.fi/~himat/WebRank/dataset_12/dataset_12/{dataset_name}"
num_pages = 85

# Stemming and Stopword Libraries for German
from nltk.stem.snowball import SnowballStemmer
import nltk
from nltk.corpus import stopwords  # Stopword library

# Initialize stemmer and stopwords for German
stemmer = SnowballStemmer(dataset_language)
nltk_german_stopwords = set(stopwords.words(dataset_language))

In [16]:
# Use spaCy's German model for lemmatization and stopword filtering
import spacy
nlp = spacy.load("de_core_news_sm")
from spacy.lang.de.stop_words import STOP_WORDS as spacy_german_stopwords

def process_keywords(keywords):
    """Apply cleaning, lemmatization, and stopword filtering to a list of keywords using spaCy's German model. Returns a processed list of useful words."""
    processed = []
    for word in keywords:
        # Clean word: keep only alphabetic characters (including German letters)
        cleaned_word = ''.join([c for c in word if c.isalpha() or c in 'äöüßÄÖÜẞ'])
        if not cleaned_word:
            continue
        doc = nlp(cleaned_word)
        for token in doc:
            lemma = token.lemma_.lower()
            if lemma and lemma not in spacy_german_stopwords:
                processed.append(lemma)
    # Return unique processed keywords (useful words only)
    return list(set(processed))

In [17]:
from bs4 import BeautifulSoup
def extract_and_process_keywords_from_tag(html_text, tag_name):
    """Extracts keywords from the content inside the specified tag in the given HTML text, processes them using process_keywords, and returns a dictionary with keyword frequencies."""
    soup = BeautifulSoup(html_text, 'html.parser')
    elements = soup.find_all(tag_name)
    raw_keywords = []
    for elem in elements:
        text = elem.get_text(separator=' ')
        raw_keywords.extend(text.split())
    processed = process_keywords(raw_keywords)
    freq = {}
    for kw in processed:
        freq[kw] = freq.get(kw, 0) + 1
    return freq

In [18]:
from bs4 import BeautifulSoup

def extract_available_tags(html_text):
    """
    Extracts and returns a list of unique HTML tag names present in the given html_text.
    """
    soup = BeautifulSoup(html_text, "html.parser")
    tags = set([tag.name for tag in soup.find_all()])
    return sorted(tags)

In [19]:
# 5. Aggregate and Print URL Ratings Across All Pages (as a pseudo-tag for sorting)
from urllib.parse import urlparse, unquote
from collections import defaultdict
import urllib.request

tag_rating_sum = defaultdict(float)
tag_count = defaultdict(int)

for i in range(num_pages):
    gt_url = f"{base_url}/{i}/GT.txt"
    web_page_url = f"{base_url}/{i}"
    url_file_url = f"{base_url}/{i}/URL.txt"
    processed_keywords = []
    headers = {"User-Agent": "Mozilla/5.0 (compatible; Copilot/1.0)"}
    gt_req = urllib.request.Request(gt_url, headers=headers)
    web_req = urllib.request.Request(web_page_url, headers=headers)
    url_req = urllib.request.Request(url_file_url, headers=headers)
    try:
        with urllib.request.urlopen(gt_req, timeout=5) as response:
            gt_text = response.read().decode("utf-8-sig").strip()
            gt_keywords = gt_text.split()
            processed_keywords = list(set(process_keywords(gt_keywords)))
    except Exception as e:
        print(f"Error processing GT for page {i}: {e}")
        continue
    # HTML tag ratings
    try:
        with urllib.request.urlopen(web_req, timeout=5) as web_response:
            html_text = web_response.read().decode("utf-8-sig").strip()
            tags = extract_available_tags(html_text)
            for tag in tags:
                result = extract_and_process_keywords_from_tag(html_text, tag)
                total_keywords = sum(result.values())
                matched_sum = sum(freq for kw, freq in result.items() if kw in processed_keywords)
                rating = matched_sum / total_keywords if total_keywords > 0 else 0
                tag_rating_sum[tag] += rating
                tag_count[tag] += 1
    except Exception as e:
        print(f"Error processing web page for page {i}: {e}")
        continue
    # URL rating as a pseudo-tag
    try:
        with urllib.request.urlopen(url_req, timeout=5) as url_response:
            real_url = url_response.read().decode("utf-8-sig").strip()
            parsed_url = urlparse(real_url)
            normalized_path = unquote(parsed_url.path.lower())
            url_tokens = re.findall(r"[a-zåäöA-ZÅÄÖ0-9]+", normalized_path)
            processed_url_keywords = process_keywords(url_tokens)
            total_url_keywords = len(processed_url_keywords)
            matched_url_keywords = sum(1 for kw in processed_url_keywords if kw in processed_keywords)
            rating = matched_url_keywords / total_url_keywords if total_url_keywords > 0 else 0
            tag_rating_sum['URL'] += rating
            tag_count['URL'] += 1
    except Exception as e:
        print(f"Error processing URL for page {i}: {e}")
        continue

# Calculate sum rating per tag and sort (including URL as a tag)
tag_sum_rating = {tag: tag_rating_sum[tag] for tag in tag_rating_sum}
sorted_tags = sorted(tag_sum_rating.items(), key=lambda x: x[1], reverse=True)

print("Tag and URL rating sum across all pages (sorted):")
for tag, rating_sum in sorted_tags:
    print(f"{tag}: {rating_sum:.4f}")

Tag and URL rating sum across all pages (sorted):
URL: 14.4500
h1: 0.3333
head: 0.3333
title: 0.3333
th: 0.2000
body: 0.1333
html: 0.1333
a: 0.0909
table: 0.0833
tr: 0.0833
hr: 0.0000
img: 0.0000
td: 0.0000
